<a href="https://colab.research.google.com/github/RIDDHI1624/Drug-Discovery/blob/main/Insulin_Receptor_Project/Phase3_step2_energy_minimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3.3 Step 2: Energy Minimization
**Steps covered:**
- Step 14: Steepest descent minimization (50,000 steps, emtol=1000 kJ/mol/nm)
- Step 15: Convergence check (final Fmax < 1000 kJ/mol/nm)

In [4]:
# Cell 1 — Extract files from zip + read n_atoms
import zipfile
import subprocess
import re
from pathlib import Path

zip_path = '/content/gromacs_system_candidate_0004.zip'

with zipfile.ZipFile(zip_path, 'r') as zf:
    for fname in ['system.gro', 'topol.top', 'complex.top']:
        if fname in zf.namelist():
            zf.extract(fname, '/content/')
            print(f'✓ Extracted {fname}')
        else:
            print(f'✗ {fname} not found in zip')

# Read n_atoms from system.gro
with open('/content/system.gro', 'r') as f:
    f.readline()
    n_atoms = int(f.readline().strip())

print(f'\n✓ n_atoms = {n_atoms}')

✓ Extracted system.gro
✓ Extracted topol.top
✓ Extracted complex.top

✓ n_atoms = 40988


In [5]:
# Cell 2 — Write energy minimization MDP file
em_mdp_content = """
; Energy minimization parameters
integrator  = steep         ; Steepest descent algorithm
emtol       = 1000.0        ; Stop when Fmax < 1000 kJ/mol/nm
emstep      = 0.01          ; Initial step size
nsteps      = 50000         ; Maximum steps

; Output
nstxout     = 500           ; Save coordinates every 500 steps
nstlog      = 500           ; Log frequency

; Neighborsearching
cutoff-scheme   = Verlet
ns_type         = grid
nstlist         = 1
rcoulomb        = 1.0
rvdw            = 1.0

; Electrostatics
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

; Bonds
constraints     = none
"""

with open('/content/em.mdp', 'w') as f:
    f.write(em_mdp_content)

print('✓ em.mdp written successfully')

✓ em.mdp written successfully


In [7]:
# Install GROMACS
import subprocess
result = subprocess.run(['apt-get', 'install', '-y', 'gromacs'],
                      capture_output=True, text=True)
print(result.stdout[-1000:])
print(result.stderr[-500:])

# Verify installation
result = subprocess.run(['gmx', '--version'], capture_output=True, text=True)
print(result.stdout[:200])
print("✓ GROMACS installed" if result.returncode == 0 else "✗ Install failed")

c link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

Processing triggers for man-db (2.10.2-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [8]:
# Cell 3 — Run grompp (preprocessing)
result = subprocess.run([
    'gmx', 'grompp',
    '-f', '/content/em.mdp',
    '-c', '/content/system.gro',
    '-p', '/content/topol.top',
    '-o', '/content/em.tpr',
    '-maxwarn', '2'
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    print('\n✓ grompp complete — em.tpr generated')
else:
    print('\n✗ grompp failed — check errors above')

Setting the LD random seed to -68698193

Generated 2145 of the 2145 non-bonded parameter combinations

Generated 2145 of the 2145 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'Protein_chain_A'

Excluding 2 bonded neighbours molecule type 'SOL'

Excluding 1 bonded neighbours molecule type 'NA'

Excluding 1 bonded neighbours molecule type 'CL'
Analysing residue names:
There are:   259    Protein residues
There are: 12264      Water residues
There are:    82        Ion residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 56x56x56, spacing 0.150 0.150 0.150

Estimate for the relative computational load of the PME mesh part: 0.17

This run will generate roughly 51 Mb of data

              :-) GROMACS - gmx grompp, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emi

In [9]:
# Cell 4 — Step 14: Run steepest descent energy minimization
result = subprocess.run([
    'gmx', 'mdrun',
    '-v',
    '-deffnm', '/content/em'
], capture_output=True, text=True)

print(result.stdout[-3000:])  # last 3000 chars to avoid overflow
print(result.stderr[-3000:])

if result.returncode == 0:
    print('\n✓ Energy minimization complete')
else:
    print('\n✗ mdrun failed — check errors above')


070
Step=  955, Dmax= 7.5e-03 nm, Epot= -6.40846e+05 Fmax= 8.36611e+03, atom= 2070
Step=  956, Dmax= 9.0e-03 nm, Epot= -6.40886e+05 Fmax= 5.50553e+03, atom= 2070
Step=  957, Dmax= 1.1e-02 nm, Epot= -6.40863e+05 Fmax= 1.10938e+04, atom= 2070
Step=  958, Dmax= 5.4e-03 nm, Epot= -6.40914e+05 Fmax= 2.79700e+03, atom= 2070
Step=  959, Dmax= 6.5e-03 nm, Epot= -6.40924e+05 Fmax= 7.21464e+03, atom= 2070
Step=  960, Dmax= 7.8e-03 nm, Epot= -6.40957e+05 Fmax= 4.74650e+03, atom= 2070
Step=  961, Dmax= 9.4e-03 nm, Epot= -6.40947e+05 Fmax= 9.64706e+03, atom= 2070
Step=  962, Dmax= 4.7e-03 nm, Epot= -6.40982e+05 Fmax= 2.45020e+03, atom= 2070
Step=  963, Dmax= 5.6e-03 nm, Epot= -6.40999e+05 Fmax= 6.14228e+03, atom= 2070
Step=  964, Dmax= 6.7e-03 nm, Epot= -6.41027e+05 Fmax= 4.21773e+03, atom= 2070
Step=  965, Dmax= 8.1e-03 nm, Epot= -6.41028e+05 Fmax= 8.17283e+03, atom= 2070
Step=  966, Dmax= 9.7e-03 nm, Epot= -6.41059e+05 Fmax= 6.73565e+03, atom= 2070
Step=  967, Dmax= 1.2e-02 nm, Epot= -6.41038e+0

In [11]:
# Cell 5 — Step 15: Convergence check (fixed)
import re
from pathlib import Path

log_path = '/content/em.log'

with open(log_path, 'r') as f:
    log_content = f.read()

# Try both formats GROMACS uses
fmax_matches = re.findall(r'Maximum force\s*=\s*([\d.e+\-]+)', log_content)
if not fmax_matches:
    fmax_matches = re.findall(r'Fmax\s*=\s*([\d.e+\-]+)', log_content)

if fmax_matches:
    final_fmax = float(fmax_matches[-1])
    print(f'Final Fmax = {final_fmax:.2f} kJ/mol/nm')
    if final_fmax < 1000:
        print('✓ CONVERGED — Fmax < 1000 kJ/mol/nm')
        print('✓ Ready for Phase 3.3 Step 3: Equilibration')
    else:
        print('✗ NOT CONVERGED — Fmax >= 1000 kJ/mol/nm')
        print('  Increase nsteps to 100000 in em.mdp and rerun')
else:
    print('Could not parse Fmax — check em.log manually')

Final Fmax = 875.57 kJ/mol/nm
✓ CONVERGED — Fmax < 1000 kJ/mol/nm
✓ Ready for Phase 3.3 Step 3: Equilibration


In [12]:
# Cell 6 — Summary + Download
from google.colab import files

print('=' * 58)
print('  PHASE 3.3 STEP 2 - ENERGY MINIMIZATION COMPLETE')
print('=' * 58)
print(f'  Step 14 - Steepest descent  : em.gro (50,000 steps)')
print(f'  Step 15 - Convergence check : Fmax = {final_fmax:.2f} kJ/mol/nm')
print('=' * 58)
print('\nNext: Phase 3.3 Step 3 - Equilibration (NVT + NPT)')

# Download minimized structure and log
files.download('/content/em.gro')
files.download('/content/em.log')
print('\n✓ Downloaded em.gro and em.log')

  PHASE 3.3 STEP 2 - ENERGY MINIMIZATION COMPLETE
  Step 14 - Steepest descent  : em.gro (50,000 steps)
  Step 15 - Convergence check : Fmax = 875.57 kJ/mol/nm

Next: Phase 3.3 Step 3 - Equilibration (NVT + NPT)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Downloaded em.gro and em.log
